# Project 2 — Student Academic Performance Analysis
**Level:** Beginner  |  **Intern:** Sohan Nandi (CSN-OC-2026-474)  |  **Domain:** Data Analysis  |  **October 2026**

## Objective & Business Questions
Compare student scores across subjects and demographic/preparation groups.

- Average math / reading / writing scores.
- Association with test-preparation completion.
- Differences across gender, lunch, parental education, race/ethnicity.
- Correlation between subject scores (cautious interpretation).

In [1]:
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def resolve_repo_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").is_dir() and (parent / "projects").is_dir():
            return parent
        if (parent / "Dataset").is_dir():
            return parent
    return Path(__file__).resolve().parents[3]

REPO_ROOT = resolve_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
def fig_path(name):
    return str(OUT_DIR / name)

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUT_DIR)

SRC_FILE = DATASET_DIR / "StudentsPerformance.csv"
print("Source file:", SRC_FILE)
assert SRC_FILE.exists(), f"Source file missing: {SRC_FILE}"
df = pd.read_csv(SRC_FILE, encoding="utf-8")
print("Loaded shape:", df.shape)


Repo root: C:\Users\sohan\OneDrive\Desktop\CyberNova
Dataset dir: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset
Output dir: outputs
Source file: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset\StudentsPerformance.csv
Loaded shape: (1000, 8)


In [2]:
print("dimensions:", df.shape)
print("columns:", list(df.columns))
print("dtypes:"); print(df.dtypes.to_string())
print("sample:"); print(df.head().to_string())
print("--- numeric stats ---")
print(df[["math score","reading score","writing score"]].describe().to_string())

dimensions: (1000, 8)
columns: ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course', 'math score', 'reading score', 'writing score']
dtypes:
gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
sample:
   gender race/ethnicity parental level of education         lunch test preparation course  math score  reading score  writing score
0  female        group B           bachelor's degree      standard                    none          72             72             74
1  female        group C                some college      standard               completed          69             90             88
2  female        group B             master's degree      standard                    none          90            

In [3]:
print("nulls:"); print(df.isnull().sum().to_string())
print("duplicate rows:", int(df.duplicated().sum()))
print("gender:", df["gender"].value_counts().to_dict())
print("race:", df["race/ethnicity"].value_counts().to_dict())
print("parental edu:", df["parental level of education"].value_counts().to_dict())
print("lunch:", df["lunch"].value_counts().to_dict())
print("prep:", df["test preparation course"].value_counts().to_dict())
for c in ["math score","reading score","writing score"]:
    print(c, "range:", df[c].min(), "-", df[c].max())

nulls:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
duplicate rows: 0
gender: {'female': 518, 'male': 482}
race: {'group C': 319, 'group D': 262, 'group B': 190, 'group E': 140, 'group A': 89}
parental edu: {'some college': 226, "associate's degree": 222, 'high school': 196, 'some high school': 179, "bachelor's degree": 118, "master's degree": 59}
lunch: {'standard': 645, 'free/reduced': 355}
prep: {'none': 642, 'completed': 358}
math score range: 0 - 100
reading score range: 17 - 100
writing score range: 10 - 100


In [4]:
clean = df.copy()
assert clean["math score"].between(0,100).all()
assert clean["reading score"].between(0,100).all()
assert clean["writing score"].between(0,100).all()
clean["overall_score"] = clean[["math score","reading score","writing score"]].mean(axis=1)
print("cleaned rows:", len(clean))
print("overall mean:", round(clean["overall_score"].mean(), 2))
clean.to_csv(OUT_DIR / "cleaned_student_performance.csv", index=False)
print("saved -> outputs/cleaned_student_performance.csv")

cleaned rows: 1000
overall mean: 67.77
saved -> outputs/cleaned_student_performance.csv


In [5]:
subs = ["math score","reading score","writing score"]
print("subject averages:"); print(df[subs].mean().round(2).to_string())
print("by test preparation:"); print(df.groupby("test preparation course")[subs].mean().round(2).to_string())
print("by gender:"); print(df.groupby("gender")[subs].mean().round(2).to_string())
print("by lunch:"); print(df.groupby("lunch")[subs].mean().round(2).to_string())
print("by race/ethnicity:"); print(df.groupby("race/ethnicity")[subs].mean().round(2).to_string())
print("by parental edu:"); print(df.groupby("parental level of education")[subs].mean().round(2).to_string())
print("correlations:"); print(df[subs].corr().round(3).to_string())

subject averages:
math score       66.09
reading score    69.17
writing score    68.05
by test preparation:
                         math score  reading score  writing score
test preparation course                                          
completed                     69.70          73.89          74.42
none                          64.08          66.53          64.50
by gender:
        math score  reading score  writing score
gender                                          
female       63.63          72.61          72.47
male         68.73          65.47          63.31
by lunch:
              math score  reading score  writing score
lunch                                                 
free/reduced       58.92          64.65          63.02
standard           70.03          71.65          70.82
by race/ethnicity:
                math score  reading score  writing score
race/ethnicity                                          
group A              61.63          64.67          62.67
g

In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
df[subs].plot(kind="box", ax=axes[0,0], patch_artist=True)
axes[0,0].set_title("Score Distribution by Subject"); axes[0,0].set_ylabel("Score")
df.groupby("test preparation course")[subs].mean().plot(kind="bar", ax=axes[0,1])
axes[0,1].set_title("Avg Scores by Test Preparation"); axes[0,1].tick_params(axis="x", rotation=0); axes[0,1].set_ylabel("Score")
df.groupby("gender")[subs].mean().plot(kind="bar", ax=axes[0,2])
axes[0,2].set_title("Avg Scores by Gender"); axes[0,2].tick_params(axis="x", rotation=0); axes[0,2].set_ylabel("Score")
clean["overall_score"].plot(kind="hist", bins=15, ax=axes[1,0], color="#2ca02c", edgecolor="white")
axes[1,0].set_title("Overall Score Distribution"); axes[1,0].set_xlabel("Overall Score")
df.groupby("lunch")[subs].mean().plot(kind="bar", ax=axes[1,1])
axes[1,1].set_title("Avg Scores by Lunch Type"); axes[1,1].tick_params(axis="x", rotation=0); axes[1,1].set_ylabel("Score")
pd.plotting.scatter_matrix(df[subs], ax=axes[1,2], figsize=(6,6), diagonal="kde", color="#9467bd", alpha=0.5)
axes[1,2].set_title("Subject Score Scatter Matrix")
fig.tight_layout(); fig.savefig(fig_path("charts_student.png"), dpi=150, bbox_inches="tight"); plt.close(fig)
print("saved outputs/charts_student.png")

saved outputs/charts_student.png


In [7]:
print("KEY FINDINGS (student):")
print("1. Math %.2f | Reading %.2f | Writing %.2f" % (
    df["math score"].mean(), df["reading score"].mean(), df["writing score"].mean()))
g = df.groupby("test preparation course")[["math score","reading score","writing score"]].mean()
print("2. Prep completed vs none -> math %.2f vs %.2f; reading %.2f vs %.2f; writing %.2f vs %.2f" % (
    g.loc["completed","math score"], g.loc["none","math score"],
    g.loc["completed","reading score"], g.loc["none","reading score"],
    g.loc["completed","writing score"], g.loc["none","writing score"]))
print("3. Gender: males higher math (%.2f vs %.2f); females higher reading (%.2f vs %.2f) and writing (%.2f vs %.2f)" % (
    df.loc[(df.gender=="male"),"math score"].mean(), df.loc[(df.gender=="female"),"math score"].mean(),
    df.loc[(df.gender=="female"),"reading score"].mean(), df.loc[(df.gender=="male"),"reading score"].mean(),
    df.loc[(df.gender=="female"),"writing score"].mean(), df.loc[(df.gender=="male"),"writing score"].mean()))
print("4. Lunch: standard %.2f vs free/reduced %.2f math" % (
    df.loc[(df.lunch=="standard"),"math score"].mean(), df.loc[(df.lunch=="free/reduced"),"math score"].mean()))
print("5. Math-reading corr: %.2f (strong positive association)" % df["math score"].corr(df["reading score"]))
print("6. Group sizes: completed %d, none %d" % (
    (df["test preparation course"]=="completed").sum(), (df["test preparation course"]=="none").sum()))
print("RECOMMENDATIONS:")
print("- Test preparation is associated with higher scores; explore expanding access (association, not causation).")
print("- Consider subject-specific support: males for reading/writing, females for math.")
print("- Students on free/reduced lunch score lower; investigate targeted tutoring/resources.")


KEY FINDINGS (student):
1. Math 66.09 | Reading 69.17 | Writing 68.05
2. Prep completed vs none -> math 69.70 vs 64.08; reading 73.89 vs 66.53; writing 74.42 vs 64.50
3. Gender: males higher math (68.73 vs 63.63); females higher reading (72.61 vs 65.47) and writing (72.47 vs 63.31)
4. Lunch: standard 70.03 vs free/reduced 58.92 math
5. Math-reading corr: 0.82 (strong positive association)
6. Group sizes: completed 358, none 642
RECOMMENDATIONS:
- Test preparation is associated with higher scores; explore expanding access (association, not causation).
- Consider subject-specific support: males for reading/writing, females for math.
- Students on free/reduced lunch score lower; investigate targeted tutoring/resources.


In [8]:
print("LIMITATIONS:")
print("- Observational: demographics and prep are associated with scores, not causal.")
print("- Group averages must not be used to predict individuals or reinforce stereotypes.")
print("- Single cross-sectional snapshot; no causation claim.")
print("CONCLUSION: Average scores sit in the mid-60s, reading and writing slightly above math. Test-preparation completion and lunch type show the largest group gaps. Any intervention must be framed as association, not causation.")

LIMITATIONS:
- Observational: demographics and prep are associated with scores, not causal.
- Group averages must not be used to predict individuals or reinforce stereotypes.
- Single cross-sectional snapshot; no causation claim.
CONCLUSION: Average scores sit in the mid-60s, reading and writing slightly above math. Test-preparation completion and lunch type show the largest group gaps. Any intervention must be framed as association, not causation.
